# Profanity Analysis: White House TikTok vs Instagram

The White House sanitized language on Instagram while allowing a more aggressive tone on TikTok. Based on my compilation (see table here - https://docs.google.com/document/d/1bTZSGqM7tkGrm7oCItLNJri7WZ8HtHbq3FwGXWdSckc/edit?usp=sharing), at least 35 of its first 600 posts contain profanities. Of these, 24 (69%) do not appear on Instagram. 15 posts contain uncensored profanities.

This notebook structures the profanity data and exports `profanity.json` for the scrollytelling visualization.

In [1]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

PROJECT = Path("/Users/wongpeiting/Desktop/CU/python-work/peak-meme")
WAR_MEMES = Path("/Users/wongpeiting/Desktop/CU/python-work/war-memes")

# Load grid_posts for cross-referencing video IDs
with open(PROJECT / "data" / "grid_posts.json") as f:
    posts = json.load(f)
posts_by_date = {}
for p in posts:
    posts_by_date.setdefault(p["date"], []).append(p)

# Full profanity table from PT's research (35 posts)
# Each entry: (date, words, bleeped, source, on_instagram, caption_hint)
PROFANITY_TABLE = [
    ("2025-08-20", ["fuck"], "bleeped", "Movie/meme clip", "yes", "Okay, f**k you, watch this"),
    ("2025-09-01", ["fuck", "shit"], "bleeped", "Step Brothers clip", "no", "We're here to f*** s*** up"),
    ("2025-09-15", ["fuck"], "obscured", "Caption/acronym (FAFO)", "yes_scrubbed", "FAFO"),
    ("2025-09-22", ["fucking"], "uncensored", "The Hangover audio clip", "no", "We're back baby, we're fucking back!"),
    ("2025-09-24", ["hell"], "uncensored", "JD Vance direct speech", "no", "You can go straight to hell"),
    ("2025-10-06", ["bullshit"], "uncensored", "AI-generated voice mocking Schumer", "yes", "Woke, trans bullshit"),
    ("2025-10-17", ["fuck"], "mixed", "Trump direct speech + caption FAFO", "no", "He doesn't want to fuck around"),
    ("2025-10-20", ["ass"], "uncensored", "Trump direct speech", "no", "I work my ass off"),
    ("2025-10-20", ["fucking"], "obscured", "Caption/acronym (LFG)", "no", "LFG!"),
    ("2025-10-24", ["ass", "bitch", "motherfucking"], "mixed", "Text overlay + rap song Victory Lap Five", "no", "Work my a** off + song lyrics"),
    ("2025-10-27", ["fucking", "shit"], "bleeped", "Wolf of Wall Street + Rick & Morty clips", "yes", "Let's f***ing move + Get your s*** together"),
    ("2025-11-06", ["fucking", "motherfucking"], "bleeped", "Mashup of TikTok posts", "no", "I'm f**king FERAL today"),
    ("2025-12-09", ["fuck", "hoes", "niggas"], "bleeped", "Money In The Grave x Last Christmas rap", "no", "Where the f*** should I even start"),
    ("2025-12-16", ["niggas"], "uncensored", "Many Men by 50 Cent", "yes_scrubbed", "Niggas tryin' to take my life away"),
    ("2025-12-27", ["shit"], "uncensored", "NOKIA by Drake", "no", "Who's callin' that shit"),
    ("2025-12-30", ["fuck"], "uncensored", "Trump direct speech", "no", "They don't know what the fuck they are doing"),
    ("2025-12-30", ["bitch", "nigga", "fuck", "ass"], "uncensored", "Up by Cardi B", "no", "Multiple expletives in full"),
    ("2025-12-31", ["fucking"], "obscured", "Caption/acronym (LFG)", "yes", "THE YEAR AMERICA LEVELED UP LFG"),
    ("2026-01-03", ["fuck"], "obscured", "Caption (FAFO)", "yes_scrubbed", "FAFO"),
    ("2026-01-04", ["fucked"], "obscured", "Pete Hegseth speech (F-ed)", "yes", "He F-ed around and he found out"),
    ("2026-01-05", ["fucking"], "mixed", "Caption LFG + bleeped voiceover", "no", "LFG + how about a cool f***ing training montage"),
    ("2026-01-09", ["fucking", "bitch"], "bleeped", "Clips of ICE protestors", "yes", "F******* ******* + F**** die b****"),
    ("2026-01-09", ["fuck", "bitches"], "mixed", "Timothee Chalamet rap remix", "no", "What the f*** + model bitches in Peckham"),
    ("2026-01-11", ["motherfucker"], "bleeped", "Movie clip", "no", "This motherf***er is locked"),
    ("2026-01-19", ["fuck"], "uncensored", "Text overlay + voiceover", "no", "FUCK IT. It doesn't exist."),
    ("2026-01-23", ["fuck"], "uncensored", "TikTok reaction post", "no", "Now what the fuck!"),
    ("2026-01-24", ["fucking"], "uncensored", "Deadpool & Wolverine clip", "no", "Let's fucking go"),
    ("2026-01-26", ["fuck"], "bleeped", "Mayor Jacob Frey speech", "yes", "Get the f*** out of Minneapolis"),
    ("2026-02-03", ["fuck", "fucking"], "uncensored", "Caption + Hell Yeah Brother song", "no", "F@$K YA BROTHER"),
    ("2026-02-06", ["fucking"], "obscured", "Caption (LFG)", "no", "RAHHH AMERICA LFG"),
    ("2026-02-19", ["fuck"], "bleeped", "Pitbull stage rant", "no", "F***k you at the same time"),
    ("2026-02-22", ["fucks"], "bleeped", "AI-doctored video", "no", "Maple eating f*cks"),
    ("2026-03-06", ["shit"], "uncensored", "GTA: San Andreas clip", "yes", "Ah shit, here we go again"),
    ("2026-03-09", ["shit"], "uncensored", "Trump direct speech", "no", "Bomb the shit out of them"),
    ("2026-03-18", ["fucking"], "bleeped", "Wolf of Wall Street clip", "no", "22 Million dollars in three Fu*king hours"),
]

print(f"Loaded {len(PROFANITY_TABLE)} profanity entries")

Loaded 35 profanity entries


In [2]:
# Match each profanity entry to a video_id from grid_posts
# Some dates have multiple posts, so we match by caption hint + expletive tag

def match_to_video_id(date, caption_hint, posts_by_date):
    """Try to match a profanity entry to a specific video_id."""
    candidates = posts_by_date.get(date, [])
    
    # First: look for posts with 'expletive' tag on that date
    expletive_posts = [p for p in candidates if "expletive" in p.get("tags", [])]
    
    if len(expletive_posts) == 1:
        return expletive_posts[0]["id"]
    
    # If multiple expletive posts on same date, try caption matching
    if len(expletive_posts) > 1:
        for p in expletive_posts:
            cap = p.get("caption", "").lower()
            hint = caption_hint.lower()[:20]
            if hint[:10] in cap or any(w in cap for w in hint.split()[:2]):
                return p["id"]
        # Return first expletive post as fallback
        return expletive_posts[0]["id"]
    
    # No expletive-tagged posts; try caption matching on all posts
    for p in candidates:
        cap = p.get("caption", "").lower()
        hint = caption_hint.lower()
        if hint[:15] in cap:
            return p["id"]
    
    # Fallback: return first post on that date
    return candidates[0]["id"] if candidates else None


profanity_records = []
for date, words, bleeped, source, on_ig, hint in PROFANITY_TABLE:
    vid = match_to_video_id(date, hint, posts_by_date)
    post = next((p for p in posts if p["id"] == vid), {}) if vid else {}
    
    profanity_records.append({
        "id": vid,
        "date": date,
        "caption": post.get("caption", hint)[:80],
        "views": post.get("views", 0),
        "words": words,
        "treatment": bleeped,  # "bleeped", "uncensored", "obscured", "mixed"
        "source": source,
        "on_instagram": on_ig,  # "yes", "no", "yes_scrubbed"
        "subject": post.get("subject", ""),
        "packaging_level": post.get("packaging_level", 0),
        "hint": hint,
    })

df = pd.DataFrame(profanity_records)
print(f"Matched {df['id'].notna().sum()} of {len(df)} entries to video IDs")
print(f"\nTreatment breakdown:")
print(df["treatment"].value_counts().to_string())
print(f"\nInstagram status:")
print(df["on_instagram"].value_counts().to_string())

# Flag unmatched
unmatched = df[df["id"].isna()]
if len(unmatched) > 0:
    print(f"\n⚠ {len(unmatched)} unmatched entries:")
    for _, row in unmatched.iterrows():
        print(f"  {row['date']}: {row['hint']}")

Matched 35 of 35 entries to video IDs

Treatment breakdown:
treatment
uncensored    14
bleeped       11
obscured       6
mixed          4

Instagram status:
on_instagram
no              24
yes              8
yes_scrubbed     3


## Summary Statistics

In [3]:
# Key stats for the scrollytelling
total = len(df)
not_on_ig = len(df[df["on_instagram"] == "no"])
scrubbed = len(df[df["on_instagram"] == "yes_scrubbed"])
on_ig_as_is = len(df[df["on_instagram"] == "yes"])
uncensored = len(df[df["treatment"] == "uncensored"])

print(f"Total posts with profanities: {total}")
print(f"  Not on Instagram: {not_on_ig} ({not_on_ig/total*100:.0f}%)")
print(f"  On Instagram but scrubbed: {scrubbed}")
print(f"  On Instagram as-is: {on_ig_as_is}")
print(f"  Uncensored on TikTok: {uncensored}")
print(f"  Bleeped: {len(df[df['treatment'] == 'bleeped'])}")
print(f"  Obscured (acronym): {len(df[df['treatment'] == 'obscured'])}")
print(f"  Mixed: {len(df[df['treatment'] == 'mixed'])}")

# Monthly trend
df["month"] = pd.to_datetime(df["date"]).dt.to_period("M")
print(f"\nMonthly profanity count:")
print(df.groupby("month").size().to_string())

Total posts with profanities: 35
  Not on Instagram: 24 (69%)
  On Instagram but scrubbed: 3
  On Instagram as-is: 8
  Uncensored on TikTok: 14
  Bleeped: 11
  Obscured (acronym): 6
  Mixed: 4

Monthly profanity count:
month
2025-08     1
2025-09     4
2025-10     6
2025-11     1
2025-12     6
2026-01    10
2026-02     4
2026-03     3
Freq: M


## Export profanity.json

In [4]:
# Build export structure
export = {
    "summary": {
        "total": total,
        "not_on_instagram": not_on_ig,
        "scrubbed_for_instagram": scrubbed,
        "on_instagram_as_is": on_ig_as_is,
        "uncensored_count": uncensored,
    },
    "posts": []
}

for _, row in df.iterrows():
    export["posts"].append({
        "id": row["id"],
        "date": row["date"],
        "caption": row["caption"],
        "views": int(row["views"]) if pd.notna(row["views"]) else 0,
        "words": row["words"],
        "treatment": row["treatment"],
        "source": row["source"],
        "on_instagram": row["on_instagram"],
        "subject": row["subject"],
        "packaging_level": int(row["packaging_level"]) if pd.notna(row["packaging_level"]) else 0,
    })

output_path = PROJECT / "data" / "profanity.json"
with open(output_path, "w") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print(f"Exported to {output_path}")
print(f"  {len(export['posts'])} posts")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

Exported to /Users/wongpeiting/Desktop/CU/python-work/peak-meme/data/profanity.json
  35 posts
  File size: 12.6 KB


## Treatment reclassifications

Four posts were originally classified as `mixed` treatment. These were resolved based on whether the actual spoken/visible word was uncensored or bleeped.

| Post # | Date | Caption | Original | Changed to | Reason |
|--------|------|---------|----------|------------|--------|
| #205 | 2025-10-17 | FAFO. | mixed | uncensored | Trump direct speech, uncensored |
| #221 | 2025-10-24 | Can't stop winning | mixed | uncensored | Rap lyrics (Victory Lap Five), uncensored in audio |
| #375 | 2026-01-05 | LFG!! | mixed | bleeped | Caption says LFG but voiceover is bleeped |
| #383 | 2026-01-09 | The far left's dangerous rhetoric on ICE | mixed | bleeped | ICE protestor footage, expletives are bleeped |

Decision rule: `mixed` resolved to whichever treatment the actual spoken/visible word received.